<a href="https://colab.research.google.com/github/jppeirce/DSC210-Foundations-of-Data-Science/blob/main/Notes/10-supervised_classification/10-supervised_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 10: Supervised Learning I, Classification

**DSC 210 Foundations of Data Science**

References:
- [Hands-on Introduction to Data Science with Python](https://florian-huber.github.io/data_science_course/) (CC BY-NC-SA 4.0)
- [scikit-learn user guide](https://scikit-learn.org/stable/supervised_learning.html), on supervised learning
- Palmer Penguins (Horst, Hill, and Gorman, 2020; CC0)

```
ASK  ->  GET  ->  EXPLORE  ->  [ MODEL ]  ->  COMMUNICATE
```

*Last major revision: 2026-08-13*

Module 9 clustered data that had no answer key. Today the answer key exists, and everything changes: we can finally measure whether the model is any good.

## Key Concepts

- Distinguish features from labels, and classification from regression
- Explain why a model must be evaluated on data it has never seen, and perform a **train/test split**
- Carry out **k-nearest-neighbours** by hand, and explain why `k` and feature scaling both change the answer
- Compute the **Gini index** and **Gini gain** by hand to choose a decision-tree split
- Read a **confusion matrix**, and compute accuracy, precision, recall, and F1
- Explain why accuracy alone is misleading on imbalanced data

---
## 1. Learning With an Answer Key
---

Last module a store asked "what kinds of customers do we have?" and no column held the answer. Today the question changes: *"which customers will respond to this campaign?"* and the store has a column recording who responded last time.

That column is a **label**, and its presence changes everything.

**Definition.** In **supervised learning** each case comes with a known label. The **features** ($X$) are the inputs used to predict, and the **label** ($y$) is what we predict. A **model** is a function that maps $X$ to $y$; **training** is the automatic search for the version of that function which fits the known labels best.

Two flavours, decided entirely by what kind of thing $y$ is:

- **Classification:** the label is a category. *Which species is this penguin? Is this call fraudulent?*
- **Regression:** the label is a number. *How long is this penguin's flipper? What will this house sell for?*

Today is classification. Module 11 takes up regression.

**A little vocabulary that will keep recurring.**

- **Parameters** are learned by the model during training. You do not choose them.
- **Hyperparameters** are settings *you* choose before training, like `k` in k-nearest-neighbours or `max_depth` in a decision tree. The model cannot learn them for you.
- **Prediction** (or inference) is applying the trained model to new cases.

The distinction between the two kinds of setting is worth holding on to. Nearly every practical difficulty in this module is a hyperparameter you have to choose without being told the right value.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/10-supervised_classification/fig_ai_vs_ml_vs_deep_learning.png?raw=true" width="520">

*Machine learning is one region of a larger territory. Everything in this module and the next sits in the middle ring: algorithms that learn rules from data rather than having them hard-coded.*

In [ ]:
# RUN-TOGETHER
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

penguins = sns.load_dataset('penguins').dropna()
print(penguins.shape)
penguins.head(3)

---
## 2. The Train/Test Split
---

Here is the central problem of supervised learning, and it is easy to state.

We want a model that works on data it has never seen. But we only have data we *have* seen. If we train a model on all of it and then measure how well it does on that same data, we learn nothing about the future: we have measured the model's memory, not its judgment.

An extreme example makes the point. A model that simply memorizes every training case and looks up the answer scores **100% on the training data** and is useless on anything new.

**Definition.** A **train/test split** holds part of the data back. The model is trained on the **training set** only, and evaluated on the **test set**, which it has never seen. The test score estimates performance on genuinely new data.

The one rule that matters: **the test set must not influence training in any way.** Not the model, not the scaling, not the choice of `k`. The moment it does, your estimate is optimistic and you have no way to tell by how much.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/10-supervised_classification/fig_train_test_split.png?raw=true" width="720">

In [ ]:
# RUN-TOGETHER
from sklearn.model_selection import train_test_split

# Features: the four body measurements. Label: the species.
X = penguins[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']]
y = penguins['species']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=210, stratify=y)

print('training cases:', X_train.shape[0])
print('test cases    :', X_test.shape[0])
print()
print('species mix, training:'); print(y_train.value_counts(normalize=True).round(3))

`test_size=0.25` holds back a quarter. `random_state` fixes the shuffle so we all get the same split. `stratify=y` keeps the species proportions the same in both halves, which matters whenever one class is rare.

> **Discuss.** Suppose you tried 40 different models and reported the best test score. In what sense has the test set now influenced your choice, and is that score still an honest estimate of future performance? (Section 5 of Module 11 gives this problem its name and its fix.)

---
## 3. k-Nearest Neighbours
---
### 3.1 The whole algorithm, in three lines

To predict the label of a new point $x$:

1. Compute the distance from $x$ to every point in the training data.
2. Keep the $k$ closest ones.
3. Predict the **majority label** among those $k$ neighbours.

That is the entire algorithm. There is no equation to solve and nothing is optimized; the "training" consists of storing the data. What makes it a *learning* algorithm is that we specify only the procedure, and the data supplies the answers.

The distance in step 1 is usually **Euclidean**:

$$d(a, b) \;=\; \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

which for two features is the ordinary straight-line distance you already know.

The same procedure serves both tasks. For **classification** the k neighbours vote; for **regression** (Module 11) their values are averaged.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/10-supervised_classification/fig_knn_algorithm.png?raw=true" width="760">

#### **Activity 10.1 - k-nearest-neighbours by hand**

Six labelled training points, two features:

| point | $x_1$ | $x_2$ | label |
| --- | --- | --- | --- |
| P1 | 4 | 4 | **A** |
| P2 | 6 | 4 | **A** |
| P3 | 3 | 5 | **A** |
| P4 | 5 | 6 | **B** |
| P5 | 10 | 10 | **B** |
| P6 | 1 | 9 | **B** |

A new point arrives: $Q = (5, 5)$. What is its label?

**Step 1. Compute the distance from Q to each point.** Use $d = \sqrt{(x_1 - 5)^2 + (x_2 - 5)^2}$. Leave answers as square roots or round to two decimals.

| point | $(x_1 - 5)^2$ | $(x_2 - 5)^2$ | $d$ | label |
| --- | --- | --- | --- | --- |
| P1 |  |  |  | A |
| P2 |  |  |  | A |
| P3 |  |  |  | A |
| P4 |  |  |  | B |
| P5 |  |  |  | B |
| P6 |  |  |  | B |

**Step 2. Sort by distance.** List the points nearest to furthest:

____ , ____ , ____ , ____ , ____ , ____

**Step 3. Vote.**

- $k = 1$: the nearest neighbour is ______, so predict ______.
- $k = 3$: the three nearest are ______, ______, ______ with labels ______, so predict ______.
- $k = 5$: predict ______.

**Questions.**

**A.** Your answers for $k = 1$ and $k = 3$ **disagree**. Explain in one sentence what the data looks like near $Q$ that causes this.

**B.** Which answer do you trust more, and why? Is there a fact about the data that would change your mind?

**C.** `k` is a hyperparameter: nothing in the data tells you its value. What would happen if you set $k = 6$ here? What does that suggest about very large `k`?

**D.** Sketch the six points and $Q$ on graph paper. Does the picture agree with your arithmetic?

#### **Class Example 10.1 - The same six points, in code**

In [ ]:
# RUN-TOGETHER
from sklearn.neighbors import KNeighborsClassifier

X_toy = np.array([[4,4],[6,4],[3,5],[5,6],[10,10],[1,9]])
y_toy = np.array(['A','A','A','B','B','B'])
Q = np.array([[5,5]])

for k in (1, 3, 5):
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_toy, y_toy)
    print(f'k = {k}:  predicts {knn.predict(Q)[0]}')

print()
dist, idx = KNeighborsClassifier(n_neighbors=6).fit(X_toy, y_toy).kneighbors(Q)
print('distances, nearest first:', dist.round(3).ravel())
print('their labels            :', y_toy[idx].ravel())

Check the distances against your table. The prediction really does flip from **B** to **A** as `k` goes from 1 to 3, and no amount of cleverness resolves which is correct: it is a choice about how much to trust the single closest point.

### 3.2 Why scaling is not optional

k-NN measures distance, and distance is measured in whatever units the columns happen to use. Look at what our penguin features actually range over.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/10-supervised_classification/fig_knn_by_hand.png?raw=true" width="520">

*The rings sit at d = 1, $\sqrt{2}$, and 2. P4 alone is inside the first ring, which is why k=1 says B; widening to three neighbours pulls in two A's.*

This flip is not a quirk of our six points. Panel **A** below shows the same thing happening between k=3 and k=5, and panel **B** shows a related hazard: a query point far outside the data still receives a confident prediction, because *some* points are always nearest.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/10-supervised_classification/fig_knn_caveats.png?raw=true" width="760">

In [ ]:
# RUN-TOGETHER
print(X_train.describe().loc[['min', 'max']].round(1))

`body_mass_g` spans thousands; `bill_depth_mm` spans about ten. In the Euclidean formula those are squared and summed together, so a 200-gram difference in mass swamps a 5-mm difference in bill depth entirely. **Without scaling, k-NN on this data is very nearly k-NN on body mass alone.**

The repair is Module 9's: standardize every column to mean 0 and standard deviation 1. But note the rule carefully.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/10-supervised_classification/fig_knn_scaling.png?raw=true" width="820">

*Q's nearest neighbour is A in raw units and B after standardizing. Nothing about the penguins changed; only the ruler did.*

In [ ]:
# RUN-TOGETHER
from sklearn.preprocessing import StandardScaler

# Fit the scaler on the TRAINING data only, then apply it to both halves.
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

print('training means after scaling:', X_train_s.mean(axis=0).round(3))
print('training sds after scaling  :', X_train_s.std(axis=0).round(3))

**Why fit the scaler on training data only?** Because the test set is supposed to be data we have never seen, and computing its mean requires seeing it. Fitting the scaler on everything leaks information from the test set into training, and the resulting score is optimistic. This mistake is extremely common and completely invisible in the output.

### 3.3 k-NN on the penguins

In [ ]:
# RUN-TOGETHER
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)
pred_knn = knn.predict(X_test_s)

print('test accuracy, scaled  :', round(knn.score(X_test_s, y_test), 3))

# For contrast: the same model on UNSCALED features.
knn_raw = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
print('test accuracy, unscaled:', round(knn_raw.score(X_test, y_test), 3))

> **Discuss.** Both numbers come from the same algorithm, the same data, and the same `k`. Only the units changed. Which of the three limitations of k-NN does this illustrate, and what would you have concluded if you had only ever run the unscaled version?

One more view of `k`, this time as a picture of the whole decision surface rather than a single prediction.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/10-supervised_classification/fig_knn_boundary_vs_k.png?raw=true" width="880">

*At k=1 the boundary contorts around individual points, including noise. At k=25 it is nearly a straight line. Neither extreme is right, and the middle is a judgment call.*

---
## 4. Decision Trees
---
### 4.1 The idea

A decision tree asks a sequence of yes/no questions about the features, and each answer sends a case down a branch until it reaches a leaf carrying a prediction. It is the one algorithm in this course whose output a non-specialist can read directly.

The only real question is: **which question should the tree ask first?** A good split separates the classes; a bad one leaves them mixed. To choose automatically, we need to measure "mixed".

### 4.2 The Gini index

**Definition.** For a node containing $K$ classes, with $p_i$ the proportion of class $i$, the **Gini index** is

$$G \;=\; 1 - \sum_{i=1}^{K} p_i^{\,2}$$

It measures **impurity**. Read the extremes to get the feel of it:

- All one class: $p_1 = 1$, so $G = 1 - 1 = 0$. Perfectly pure.
- Two classes evenly split: $p_1 = p_2 = 0.5$, so $G = 1 - (0.25 + 0.25) = 0.5$. Maximally mixed, for two classes.

A split is good when it produces children that are purer than the parent. Since a split makes two children, we compare the parent to the *weighted average* of the two.

**Definition.** The **Gini gain** of a split is

$$\Delta G \;=\; G_{\text{parent}} \;-\; \left( \frac{n_L}{n} G_L + \frac{n_R}{n} G_R \right)$$

where $n_L$ and $n_R$ are the numbers of cases sent left and right. **The tree tries every candidate split and keeps the one with the largest gain.**

Two candidate splits on the same twenty shapes. The left column shows what each threshold does to the mix of classes; the right shows the same points in feature space.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/10-supervised_classification/fig_decision_tree_ginigain.png?raw=true" width="820">

*Neither split is pure, but one leaves the children less mixed than the other. Gini gain is how we make "less mixed" a number.*

#### **Activity 10.2 - Choosing a split by hand**

Eight penguins, sorted by bill length. Two species: **A** (Adelie) and **G** (Gentoo).

| bill length (mm) | 35 | 37 | 39 | 41 | 43 | 45 | 47 | 49 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| species | A | A | A | G | G | A | G | G |

We will evaluate two candidate splits and pick the better one.

**Step 1. The parent node.** All eight penguins are in it.

- count of A = ______   count of G = ______
- $p_A$ = ______   $p_G$ = ______
- $G_{\text{parent}} = 1 - (p_A^2 + p_G^2)$ = ______

**Step 2. Split 1: bill length < 40.**

| | goes left (< 40) | goes right ($\geq$ 40) |
| --- | --- | --- |
| which penguins? |  |  |
| count A |  |  |
| count G |  |  |
| $n$ |  |  |
| Gini |  |  |

Weighted child Gini = $\frac{n_L}{8} G_L + \frac{n_R}{8} G_R$ = ______

Gini gain = ______

**Step 3. Split 2: bill length < 42.**

| | goes left (< 42) | goes right ($\geq$ 42) |
| --- | --- | --- |
| which penguins? |  |  |
| count A |  |  |
| count G |  |  |
| $n$ |  |  |
| Gini |  |  |

Weighted child Gini = ______   Gini gain = ______

**Step 4. Which split would the tree choose?** ______

**Questions.**

**A.** One of your two splits produced a child with Gini exactly 0. What does that mean about the penguins in it?

**B.** Split 2 puts the boundary only 2 mm away from Split 1, yet the gains differ substantially. Looking at the table, which single penguin is responsible?

**C.** The penguin at 45 mm is an A sitting among Gs. No threshold on bill length alone can fix that. Name two things the tree could do next, given that it is allowed to keep splitting.

**D.** Gini is 0 for a pure node and 0.5 for an evenly mixed two-class node. What would the Gini be for a node with three classes in equal proportion? Is it still capped at 0.5?

#### **Class Example 10.2 - The same eight penguins, in code**

In [ ]:
# RUN-TOGETHER
def gini(n_a, n_g):
    n = n_a + n_g
    if n == 0: return 0.0
    return 1 - ((n_a/n)**2 + (n_g/n)**2)

bill    = [35, 37, 39, 41, 43, 45, 47, 49]
species = ['A','A','A','G','G','A','G','G']

parent = gini(species.count('A'), species.count('G'))
print(f'parent Gini = {parent:.4f}\n')

for thr in (40, 42):
    L = [s for b, s in zip(bill, species) if b <  thr]
    R = [s for b, s in zip(bill, species) if b >= thr]
    gL, gR = gini(L.count('A'), L.count('G')), gini(R.count('A'), R.count('G'))
    weighted = len(L)/8*gL + len(R)/8*gR
    print(f'split at {thr}:  left {L} Gini={gL:.4f} | right {R} Gini={gR:.4f}')
    print(f'            weighted = {weighted:.4f}   GAIN = {parent - weighted:.4f}\n')

### 4.3 Growing a tree on the penguins

Trees need no scaling: a threshold on grams does not care that another column is in millimetres. This is a real practical advantage over k-NN.

In [ ]:
# RUN-TOGETHER
from sklearn.tree import DecisionTreeClassifier, plot_tree

tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
pred_tree = tree.predict(X_test)

print('test accuracy:', round(tree.score(X_test, y_test), 3))

fig, ax = plt.subplots(figsize=(13, 6))
plot_tree(tree, feature_names=list(X.columns), class_names=tree.classes_,
          filled=True, rounded=True, fontsize=8)
plt.show()

Read the top box. It reports the feature and threshold chosen, the `gini` of the node, how many samples reached it, and the class counts. **That `gini` value is the quantity you computed by hand in Activity 10.2**, and the split shown is the one with the largest gain among all features and all thresholds.

### 4.4 The tree's characteristic failure: overfitting

`max_depth=3` was a choice. What happens without one?

A tree's predictions always look the same in feature space: **rectangles**. Every split is a vertical or horizontal cut, so the regions are boxes, never curves or diagonals.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/10-supervised_classification/fig_tree_partition.png?raw=true" width="880">

*Compare this to the k-NN boundary above. Two algorithms, two very different vocabularies of shape.*

In [ ]:
# RUN-TOGETHER
for depth in [1, 2, 3, 5, None]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_train, y_train)
    print(f'max_depth={str(depth):5}  train={t.score(X_train, y_train):.3f}  test={t.score(X_test, y_test):.3f}')

Watch the two columns separate. With no depth limit the tree drives **training** accuracy to 1.000 by growing branches until every training penguin sits in its own pure leaf. Test accuracy does not follow.

**Definition.** A model **overfits** when it learns detail specific to the training data that does not generalize. Its symptom is exactly what you see here: training score far above test score.

This is why the train/test split from Section 2 is not bureaucracy. Without it, the unlimited tree looks perfect.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/10-supervised_classification/fig_overfitting_depth.png?raw=true" width="760">

*Test accuracy peaks at depth 5 and then slips, while training accuracy climbs to a perfect 1.000. The widening shaded gap is the overfitting. Note how modest the penalty is here: the penguins are an easy, clean dataset. On messier data the test curve can fall off sharply, and the gap is your only warning.*

---
## 5. Evaluating a Classifier
---
### 5.1 Accuracy, and why it lies

Suppose a bank builds a model to flag fraudulent phone calls. Out of 1,000 calls, 50 are fraud and 950 are legitimate. The model reports **97% accuracy**.

Before reading on: is that good?

Here is the model to beat. A "model" that ignores every feature and always predicts *legitimate* gets 950 of 1,000 right, which is **95% accuracy**, and it catches no fraud at all.

**Never report an accuracy without a baseline.** The right question is never "is 97% high?" but "what would nothing at all have scored?"

### 5.2 The confusion matrix

To see what a classifier is actually doing, break its predictions into four cells.

| | predicted fraud | predicted legitimate |
| --- | --- | --- |
| **actually fraud** | True Positive (TP) | False Negative (FN) |
| **actually legitimate** | False Positive (FP) | True Negative (TN) |

From those four numbers:

$$\text{accuracy} = \frac{TP + TN}{TP + TN + FP + FN} \qquad \text{precision} = \frac{TP}{TP + FP} \qquad \text{recall} = \frac{TP}{TP + FN}$$

$$F_1 = 2 \cdot \frac{\text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$$

In words, and this is the part worth memorizing:

- **Precision:** of everything we flagged, how much was really fraud? (Low precision means crying wolf.)
- **Recall:** of all the real fraud, how much did we catch? (Low recall means missing it.)
- **F1:** a single number balancing the two, useful when classes are imbalanced.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/10-supervised_classification/fig_confusion_anatomy.png?raw=true" width="700">

*The same four numbers you are about to work with, laid out with the two metrics that come from them. Notice that precision reads down a column and recall reads across a row.*

#### **Activity 10.3 - Reading a confusion matrix by hand**

The bank's fraud model produced these results on 1,000 calls:

| | predicted fraud | predicted legitimate |
| --- | --- | --- |
| **actually fraud** | 30 | 20 |
| **actually legitimate** | 10 | 940 |

**Step 1. Identify the four cells.**

TP = ______   FN = ______   FP = ______   TN = ______

**Step 2. Compute, to three decimals.**

- accuracy = ______
- precision = ______
- recall = ______
- $F_1$ = ______

**Step 3. The baseline.** For the model that always predicts *legitimate*:

- accuracy = ______
- recall = ______

**Questions.**

**A.** Compare the model's accuracy to the baseline's. How much did all that modelling buy?

**B.** The model's recall and precision are quite different from each other. Which one is worse, and state in plain English what that failure means for the bank's customers.

**C.** The bank can lower the threshold so the model flags more calls. Predict what happens to precision and to recall, and explain why they move in opposite directions.

**D.** For fraud detection, which error is more costly: a false positive (a legitimate call flagged) or a false negative (fraud missed)? Now answer the same question for a spam filter, and for a cancer screening test. Does one answer serve all three?

#### **Class Example 10.3 - Confusion matrices in code**

In [ ]:
# RUN-TOGETHER
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test, pred_tree, labels=tree.classes_)

sns.heatmap(cm, annot=True, fmt='d', cbar=False, cmap='Blues',
            xticklabels=tree.classes_, yticklabels=tree.classes_)
plt.xlabel('predicted'); plt.ylabel('actual')
plt.title('Decision tree on the penguin test set')
plt.show()

print(classification_report(y_test, pred_tree))

`classification_report` gives precision, recall, and F1 **per class**, which matters: a model can be excellent on the common class and poor on a rare one, and a single overall accuracy hides that completely. The `support` column reports how many test cases each class actually had, and small support means the numbers beside it are jumpy.

> **Discuss.** Compare the per-class recall for the three species. Is the model equally good at all three? Which species would you want more training data for, and how did you decide?

---
## 6. Choosing a Classifier
---

| | k-nearest-neighbours | Decision tree |
| --- | --- | --- |
| Needs feature scaling | **Yes**, critically | No |
| Interpretable | Not really | **Yes**, you can read it |
| Main hyperparameter | `k` | `max_depth` |
| Training cost | none (stores data) | moderate |
| Prediction cost | high (compares to everything) | very low |
| Typical failure | wrong `k`, unscaled features | overfitting |

Neither is "better". They fail differently, and knowing how a method fails is more useful than knowing its accuracy on someone else's dataset.

**What carries forward from today**, regardless of algorithm:

1. Split before you model, and never let the test set influence training.
2. Every hyperparameter is a decision you must justify, not a default to accept.
3. Report accuracy against a baseline, or do not report it.
4. A gap between training and test score is the signature of overfitting.

Module 11 takes up **regression**, where the label is a number, and returns to evaluation with cross-validation, a better answer to "which hyperparameter?" than the guessing we did today.

## Suggested Exercises

1. Five labelled training points, one feature: $x = 1$ (**A**), $x = 2$ (**A**), $x = 5$ (**B**), $x = 6$ (**B**), $x = 7$ (**B**). A new point arrives at $x = 4$.

    a. Compute the distance from the new point to each training point.

    b. Predict its label at $k = 1$, $k = 3$, and $k = 5$.

    c. At $k = 5$ every training point votes. Explain why that prediction would be the same no matter where the new point sat.

    d. A sixth training point is added at $x = 100$ with label **A**. Which of your three predictions change? What does that say about k-NN and outliers?

2. A node contains 10 cases: 6 of class X and 4 of class Y.

    a. Compute its Gini index.

    b. A candidate split sends {5 X, 1 Y} left and {1 X, 3 Y} right. Compute both child Ginis, the weighted average, and the Gini gain.

    c. A second candidate sends {3 X, 2 Y} left and {3 X, 2 Y} right. Compute its gain without doing much arithmetic, and explain how you knew.

    d. Explain why Gini gain can never be negative.

3. A medical test for a rare disease is evaluated on 10,000 patients. 100 have the disease. The test produces 80 true positives, 20 false negatives, 400 false positives, and 9,500 true negatives.

    a. Build the confusion matrix and compute accuracy, precision, recall, and F1.

    b. What accuracy would "always predict healthy" achieve?

    c. The test's precision is low. Explain, in terms a patient would understand, what that means for someone who receives a positive result.

    d. The manufacturer advertises "95% accurate." Is that claim false? Is it honest? Distinguish the two.

4. You train a decision tree with no depth limit and get 100% training accuracy and 71% test accuracy. A classmate proposes fixing this by collecting the test set into the training set, "so the model sees more data."

    a. Name the phenomenon the two scores reveal.

    b. Explain what would happen to the two reported numbers under your classmate's proposal, and why the result would be worse than useless.

    c. Give two changes that would genuinely help.

    d. You now try `max_depth` of 1 through 10 and report the best test accuracy. Explain the subtler problem with this procedure, using the rule from Section 2.